<a href="https://colab.research.google.com/github/vallepumanik22/RecomSys/blob/main/data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
import gzip, json

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
PROJECT = "/content/drive/MyDrive/recomSys"
for sub in ["data", "reports/figures"]:
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)
print("Ready:", os.listdir(PROJECT))

Ready: ['data', 'reports']


In [3]:
!mkdir -p /content/raw
!wget -q --show-progress -P /content/raw https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz
!wget -q --show-progress -P /content/raw https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz
!ls -lh /content/raw

Video_Games.jsonl.g 100%[===================>] 776.49M  29.3MB/s    in 22s     
meta_Video_Games.js 100%[===================>]  98.32M  57.9MB/s    in 1.7s    
total 875M
-rw-r--r-- 1 root root  99M Jan 16  2025 meta_Video_Games.jsonl.gz
-rw-r--r-- 1 root root 777M Jan 17  2025 Video_Games.jsonl.gz


In [6]:
def peek(path, n=1):
    with gzip.open(path, "rt") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(json.dumps(json.loads(line), indent=2)[:3000])

print("=== REVIEW ===");   peek("/content/raw/Video_Games.jsonl.gz")
print("=== METADATA ==="); peek("/content/raw/meta_Video_Games.jsonl.gz")

=== REVIEW ===
{
  "rating": 4.0,
  "title": "It\u2019s pretty sexual. Not my fav",
  "text": "I\u2019m playing on ps5 and it\u2019s interesting.  It\u2019s unique, massive, and has a neat story.  People are freaking out angry about this game.  I don\u2019t think it\u2019s a top 10 game but it\u2019s definitely a good game on ps5 (played at launch).",
  "images": [],
  "asin": "B07DJWBYKP",
  "parent_asin": "B07DK1H3H5",
  "user_id": "AGCI7FAH4GL5FI65HYLKWTMFZ2CQ",
  "timestamp": 1608186804795,
  "helpful_vote": 0,
  "verified_purchase": true
}
=== METADATA ===
{
  "main_category": "Video Games",
  "title": "Dash 8-300 Professional Add-On",
  "average_rating": 5.0,
  "rating_number": 1,
  "features": [
    "Features Dash 8-300 and 8-Q300 ('Q' rollout livery)",
    "Airlines - US Airways, South African Express, Bahamasair, Augsburg Airways, Lufthansa Cityline, British Airways (Union Jack), British European, FlyBe, Intersky, Wideroe, Iberia, Tyrolean, QantasLink, BWIA",
    "Airports inc

In [7]:
import pandas as pd

REV  = f"{PROJECT}/data/reviews.parquet"
META = f"{PROJECT}/data/meta.parquet"

def to_float(x):
    """Prices come as numbers, strings like '$19.99', or missing. Convert safely."""
    try:
        return float(str(x).replace("$", "").replace(",", ""))
    except (TypeError, ValueError):
        return None

if not os.path.exists(REV):
    rows = []
    with gzip.open("/content/raw/Video_Games.jsonl.gz", "rt") as f:
        for line in f:
            d = json.loads(line)
            rows.append((d["user_id"], d["asin"], d["parent_asin"], d["rating"],
                         d["timestamp"], d["verified_purchase"], d["helpful_vote"],
                         len(d.get("text") or "")))
    df = pd.DataFrame(rows, columns=["user_id", "asin", "item_id", "rating", "ts",
                                     "verified", "helpful_votes", "text_len"])
    del rows                                   # free the RAM used by the Python list
    df["ts"] = pd.to_datetime(df["ts"], unit="ms")
    df["rating"] = df["rating"].astype("float32")
    df.to_parquet(REV, index=False)
    print("reviews built")

if not os.path.exists(META):
    meta_rows = []
    with gzip.open("/content/raw/meta_Video_Games.jsonl.gz", "rt") as f:
        for line in f:
            d = json.loads(line)
            imgs = d.get("images") or []
            text_parts = ([d.get("title") or ""] + (d.get("features") or [])
                          + (d.get("description") or []))
            meta_rows.append({
                "item_id": d["parent_asin"],
                "title": d.get("title") or "",
                "main_category": d.get("main_category"),
                "categories": " > ".join(d.get("categories") or []),
                "store": d.get("store"),
                "price": to_float(d.get("price")),
                "avg_rating_2023": d.get("average_rating"),
                "n_ratings_2023": d.get("rating_number"),
                "image": imgs[0].get("large") if imgs else None,
                "text": " ".join(text_parts)[:2000],
            })
    meta = pd.DataFrame(meta_rows)
    meta.to_parquet(META, index=False)
    print("meta built")

df   = pd.read_parquet(REV)
meta = pd.read_parquet(META)
print(df.shape, meta.shape)

reviews built
meta built
(4624615, 8) (137269, 10)


In [8]:
print("=== REVIEWS ===")
print(df.dtypes)
print(f"memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print("missing values:\n", df.isna().sum())
print(df.head(3))
print("rating range:", df.rating.min(), "to", df.rating.max())
print("time range:", df.ts.min(), "to", df.ts.max())

print("\n=== METADATA ===")
print("duplicate item_ids in meta:", meta.item_id.duplicated().sum())
print("reviewed products that have metadata:",
      f"{df.item_id.drop_duplicates().isin(meta.item_id).mean():.1%}")
print(meta.sample(5, random_state=0)[["title", "categories", "price"]])

=== REVIEWS ===
user_id                  object
asin                     object
item_id                  object
rating                  float32
ts               datetime64[ns]
verified                   bool
helpful_votes             int64
text_len                  int64
dtype: object
memory: 1.04 GB
missing values:
 user_id          0
asin             0
item_id          0
rating           0
ts               0
verified         0
helpful_votes    0
text_len         0
dtype: int64
                        user_id        asin     item_id  rating  \
0  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B07DJWBYKP  B07DK1H3H5     4.0   
1  AGCI7FAH4GL5FI65HYLKWTMFZ2CQ  B00ZS80PC2  B07SRWRH5D     5.0   
2  AGXVBIUFLFGMVLATYXHJYL4A5Q7Q  B01FEHJYUU  B07MFMFW34     5.0   

                       ts  verified  helpful_votes  text_len  
0 2020-12-17 06:33:24.795      True              0       226  
1 2020-04-16 15:31:54.941     False              1        85  
2 2017-03-30 12:37:11.000      True              0        